# 2. Model and prompting-method selection

Calculate mean triplet-quality scores for each model and prompting method. Compare
the observed scores with the recorded choice used in notebook 3.


In [1]:
import json

from pandas import DataFrame, set_option

from warbias_kit.dataset_workflow import create_workspace, find_project, load_inputs
from warbias_kit.io import write_json
from warbias_kit.notebook_display import display

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = create_workspace("02_model_selection")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)

## Rating records and coverage

Three annotators rated 48 pilot triplets on a 1–5 quality scale, giving 144 scores.
Each row links one score to its recorded condition and blind item ID.


In [2]:
ratings = DataFrame(inputs["pilot_human_ratings"])
ratings["score"] = ratings["score"].astype(float)
display(ratings[["blind_id", "candidate_id", "strategy", "rater_id", "score"]].head(12))
display(
    ratings.groupby("rater_id")
    .agg(items=("blind_id", "nunique"), ratings=("score", "size"))
    .reset_index()
)

,blind_id,candidate_id,strategy,rater_id,score
0,WBV-0951233DD6,lapa12b,sequential,rater_01,5.0
1,WBV-0951233DD6,lapa12b,sequential,rater_02,5.0
2,WBV-0951233DD6,lapa12b,sequential,rater_03,4.0
3,WBV-E69D51AECC,mamay27b,zero_shot,rater_01,3.0
4,WBV-E69D51AECC,mamay27b,zero_shot,rater_02,5.0
5,WBV-E69D51AECC,mamay27b,zero_shot,rater_03,4.0
6,WBV-5F8CEC8B13,lapa12b,sequential,rater_01,4.0
7,WBV-5F8CEC8B13,lapa12b,sequential,rater_02,5.0
8,WBV-5F8CEC8B13,lapa12b,sequential,rater_03,5.0
9,WBV-52435814E5,mamay27b,zero_shot,rater_01,3.0


,rater_id,items,ratings
0,rater_01,48,48
1,rater_02,48,48
2,rater_03,48,48


In [3]:
from warbias_kit.dataset_workflow import recorded_selection

analysis, selection_record = recorded_selection(project, inputs)
columns = [
    "rank",
    "candidate_id",
    "strategy",
    "generated_items",
    "rated_items",
    "ratings",
    "mean_score",
]
display(DataFrame(analysis["condition_summary"])[columns])
analysis["coverage"]

,rank,candidate_id,strategy,generated_items,rated_items,ratings,mean_score
0,1,lapa12b,sequential,13,12,36,4.388889
1,2,lapa12b,zero_shot,12,12,36,4.333333
2,3,mamay27b,sequential,12,12,36,4.111111
3,4,mamay27b,zero_shot,13,12,36,3.777778


{'generated_items': 50,
 'rated_items': 48,
 'ratings': 144,
 'raters': 3,
 'expected_ratings': 150,
 'missing_condition_ids': ['ssuk_idp_return_integration_choice_case_001__idp_current__neutral__unspecified::lapa12b::sequential',
  'ssuk_idp_employment_competence_reliability_case_005__idp_current__neutral__unspecified::mamay27b::zero_shot']}

For each condition, divide the sum of recorded scores by the number of ratings.
Lapa sequential has the highest observed mean: 158 / 36 = 4.3889. Lapa zero-shot
scores 156 / 36 = 4.3333.

The recorded author choice is Lapa with sequential prompting for separate
control of the three sentence roles.


In [4]:
write_json(work / "rating-analysis.json", analysis)
write_json(work / "selected-condition.json", selection_record)
result = {
    "stage": "model_selection",
    "selection_record_path": str(work / "selected-condition.json"),
    "selection": analysis["recorded_selection"],
    "highest_mean_conditions": analysis["highest_mean_conditions"],
    "coverage": analysis["coverage"],
}
result

{'stage': 'model_selection',
 'selection_record_path': 'runs/anonymous-final/02_model_selection-dnrjp_fx/selected-condition.json',
 'selection': {'candidate_id': 'lapa12b',
  'strategy': 'sequential',
  'mean_score': 4.388888888888889,
  'rank': 1,
  'record_type': 'reconstructed_from_contemporaneous_author_instructions_and_frozen_run'},
 'highest_mean_conditions': [{'candidate_id': 'lapa12b',
   'strategy': 'sequential',
   'generated_items': 13,
   'rated_items': 12,
   'ratings': 36,
   'mean_score': 4.388888888888889,
   'missing_condition_ids': ['ssuk_idp_return_integration_choice_case_001__idp_current__neutral__unspecified::lapa12b::sequential'],
   'rank': 1}],
 'coverage': {'generated_items': 50,
  'rated_items': 48,
  'ratings': 144,
  'raters': 3,
  'expected_ratings': 150,
  'missing_condition_ids': ['ssuk_idp_return_integration_choice_case_001__idp_current__neutral__unspecified::lapa12b::sequential',
   'ssuk_idp_employment_competence_reliability_case_005__idp_current__neut

The saved `selected-condition.json` binds the choice to the exact ratings and
pilot inputs with SHA256 hashes. Pass its relative path to `SELECTION_RECORD_PATH`
in notebook 3.
